# 🎯 YOLOv8 Object Detection on COCO8

A compact, portfolio-friendly object detection project using **Ultralytics YOLOv8** and the tiny **COCO8** dataset.

## Table of Contents
1. Object Detection Concepts
2. YOLO Pipeline
3. Environment Setup
4. Train YOLOv8 on COCO8
5. Validation and mAP
6. Inference on Images
7. Prediction vs Ground Truth
8. Limitations and Future Improvements
9. Key Learnings, Interview Questions, Common Mistakes, Further Reading


## 1. Object Detection Concepts

Object detection predicts both **what** objects are present and **where** they are located.

- **Bounding box:** rectangle around an object, usually represented as `(x, y, width, height)` or corner coordinates.
- **Confidence score:** how certain the model is about a detection.
- **IoU:** intersection-over-union, measuring box overlap quality.
- **NMS:** non-maximum suppression, removing duplicate boxes for the same object.
- **mAP:** mean average precision, summarizing detection quality across classes and IoU thresholds.

```mermaid
flowchart LR
    A[Input image] --> B[YOLO backbone]
    B --> C[Neck: feature fusion]
    C --> D[Detection head]
    D --> E[Boxes + classes + confidence]
    E --> F[NMS]
```


In [ ]:
# Reproducibility and common imports
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
try:
    from ultralytics import YOLO
    import cv2
except ImportError:
    %pip install -q ultralytics opencv-python
    from ultralytics import YOLO
    import cv2


## 2. Load YOLO Model

`yolov8n.pt` is the nano model: small enough for fast experiments, still representative of the YOLO workflow.


In [ ]:
model = YOLO("yolov8n.pt")
model.info()


## 3. Train on COCO8

COCO8 is intentionally tiny. It is ideal for demonstrating the pipeline quickly, not for producing a production detector.


In [ ]:
results = model.train(
    data="coco8.yaml",
    epochs=10,
    imgsz=640,
    batch=8,
    seed=SEED,
    project="runs/detect",
    name="coco8_portfolio",
    exist_ok=True,
)


## 4. Validate Model and Calculate mAP


In [ ]:
metrics = model.val(data="coco8.yaml", imgsz=640)
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("mAP75:", metrics.box.map75)


## 5. Run Inference on Custom Images

The example uses validation images downloaded by Ultralytics. You can replace `source` with your own image path or URL.


In [ ]:
trained_model = YOLO("runs/detect/coco8_portfolio/weights/best.pt")
predictions = trained_model.predict(source="https://ultralytics.com/images/bus.jpg", conf=0.25, save=True)
for pred in predictions:
    print(pred.boxes)


In [ ]:
def show_yolo_result(result):
    plotted = result.plot()
    plotted = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 8))
    plt.imshow(plotted)
    plt.axis("off")
    plt.title("YOLOv8 Predictions")

show_yolo_result(predictions[0])


## 6. Compare Predictions with Ground Truth

Ultralytics stores train/validation samples in the dataset directory. The validation plots generated during training are useful artifacts for comparing predictions and labels.


In [ ]:
from IPython.display import Image, display

candidate_plots = [
    Path("runs/detect/coco8_portfolio/val_batch0_labels.jpg"),
    Path("runs/detect/coco8_portfolio/val_batch0_pred.jpg"),
]
for plot_path in candidate_plots:
    if plot_path.exists():
        display(Image(filename=str(plot_path)))
    else:
        print("Missing plot:", plot_path)


## Limitations and Future Improvements

- COCO8 is too small to represent real deployment performance.
- More epochs and stronger augmentation can improve results.
- Domain-specific datasets are required for specialized tasks such as road signs or hard hats.
- Deployment should include confidence calibration and failure-case review.

## Key Learnings

- Object detection combines classification and localization.
- IoU measures box quality; mAP summarizes performance over thresholds.
- NMS removes duplicate detections.
- YOLOv8 provides a production-like training and inference workflow with little boilerplate.

## Interview Questions

1. What is the difference between classification and object detection?
2. Why does NMS improve final predictions?
3. What does mAP50 mean compared with mAP50-95?
4. Why is COCO8 unsuitable for final benchmarking?

## Common Mistakes

- Reporting only visual examples without validation metrics.
- Confusing confidence with localization quality.
- Training on tiny data and overclaiming generalization.
- Forgetting to inspect false positives and false negatives.

## Further Reading

- Ultralytics YOLO docs: https://docs.ultralytics.com/
- COCO metrics overview: https://cocodataset.org/#detection-eval
- YOLOv8 tasks: https://docs.ultralytics.com/tasks/detect/
